In [5]:
import sys
sys.path.append("..")
import os
import SimpleITK as sitk
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

from src.filtering import anisotropic_filter


In [3]:
img = sitk.ReadImage("/media/mrsmile/IA/tesis/data/processed/resampled/Diseased_4.nii.gz")

for it in [3, 5, 7]:
    for c in [1.0, 1.5, 2.0]:
        img_f = anisotropic_filter(img, iterations=it, conductance=c)
        sitk.WriteImage(img_f, f"test_it{it}_c{c}.nii.gz")

In [6]:
INPUT_DIR = "../data/processed/resampled"
OUTPUT_DIR = "../data/processed/filtered"

ITERATIONS = 3
TIMESTEP = 0.02
CONDUCTANCE = 1.5

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
def process_one(path):
    try:
        name = os.path.basename(path)
        out_path = os.path.join(OUTPUT_DIR, name)

        if os.path.exists(out_path):
            return

        img = sitk.ReadImage(path)

        img_f = anisotropic_filter(
            img,
            iterations=ITERATIONS,
            timestep=TIMESTEP,
            conductance=CONDUCTANCE
        )

        sitk.WriteImage(img_f, out_path)

    except Exception as e:
        print("Error:", path)
        print(e)


In [ ]:
paths = sorted([os.path.join(INPUT_DIR, f) for f in os.listdir(INPUT_DIR)])

workers = max(1, mp.cpu_count() - 2)
print(f"Using {workers} workers")

with ProcessPoolExecutor(max_workers=workers) as exe:
    list(tqdm(exe.map(process_one, paths), total=len(paths)))

In [4]:
import napari


img = sitk.ReadImage("/media/mrsmile/IA/tesis/data/processed/resampled/Diseased_4.nii.gz")
img_f = sitk.ReadImage("test_it5_c1.5.nii.gz")

napari.view_image(sitk.GetArrayFromImage(img), name="original")
napari.view_image(sitk.GetArrayFromImage(img_f), name="filtered")


/tmp/ipykernel_60133/3349181744.py:7: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  napari.view_image(sitk.GetArrayFromImage(img), name="original")
/tmp/ipykernel_60133/3349181744.py:8: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  napari.view_image(sitk.GetArrayFromImage(img_f), name="filtered")


Viewer(camera=Camera(center=(0.0, np.float64(312.0), np.float64(312.0)), zoom=np.float64(0.9119999999999999), angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(np.float64(116.0), 1.0, 0.0), viewbox=None, scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=1.0), dims=Dims(ndim=3, ndisplay=2, order=(0, 1, 2), axis_labels=('0', '1', '2'), rollable=(True, True, True), range=(RangeTuple(start=np.float64(0.0), stop=np.float64(232.0), step=np.float64(1.0)), RangeTuple(start=np.float64(0.0), stop=np.float64(624.0), step=np.float64(1.0)), RangeTuple(start=np.float64(0.0), stop=np.float64(624.0), step=np.float64(1.0))), margin_left=(0.0, 0.0, 0.0), margin_right=(0.0, 0.0, 0.0), point=(np.float64(116.0), np.float64(312.0), np.float64(312.0)), last_used=0), grid=GridCanvas(stride=1, shape=(-1, -1), ena